In [1]:
import numpy as np

class Conv2DNumPy:
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int, stride: int = 1, padding: int = 0):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # 随机初始化权重与偏置 (C_out, C_in, K_H, K_W)
        self.weights = np.random.randn(out_channels, in_channels, kernel_size, kernel_size).astype(np.float32) * 0.1
        self.bias = np.zeros(out_channels, dtype=np.float32)

    def forward(self, x: np.ndarray) -> np.ndarray:
        N, C_in, H, W = x.shape
        K = self.kernel_size
        S = self.stride
        P = self.padding
        
        # 1. 计算输出特征图尺寸
        H_out = (H + 2 * P - K) // S + 1
        W_out = (W + 2 * P - K) // S + 1
        
        # 2. 图像 Padding 填充
        if P > 0:
            x_padded = np.pad(x, ((0, 0), (0, 0), (P, P), (P, P)), mode='constant', constant_values=0)
        else:
            x_padded = x

        # 3. 初始化输出张量
        out = np.zeros((N, self.out_channels, H_out, W_out), dtype=np.float32)

        # 4. 显式六重循环 (N -> C_out -> H_out -> W_out -> C_in -> K_H -> K_W)
        for n in range(N):                           # 遍历 Batch 批次
            for c_out in range(self.out_channels):   # 遍历每个输出通道(卷积核)
                for h in range(H_out):               # 遍历输出特征图的高
                    for w in range(W_out):           # 遍历输出特征图的宽
                        
                        # 找到当前窗口在输入图上的起始物理坐标
                        h_start = h * S
                        w_start = w * S
                        
                        pixel_sum = 0.0
                        for c_in in range(C_in):     # 遍历输入通道
                            for kh in range(K):      # 遍历卷积核高
                                for kw in range(K):  # 遍历卷积核宽
                                    val = x_padded[n, c_in, h_start + kh, w_start + kw]
                                    w_val = self.weights[c_out, c_in, kh, kw]
                                    pixel_sum += val * w_val
                        
                        # 加上偏置
                        out[n, c_out, h, w] = pixel_sum + self.bias[c_out]
                        
        return out

class ReLUNumPy:
    def forward(self, x: np.ndarray) -> np.ndarray:
        # 元素级非线性激活 max(0, x)
        return np.maximum(0, x)

In [2]:
if __name__ == "__main__":
    np.random.seed(42)
    
    # 构造输入：Batch=1, Channels=1, Height=5, Width=5
    x_input = np.random.randn(1, 1, 5, 5).astype(np.float32)
    print("=== 输入张量 X Shape ===")
    print(x_input.shape)
    
    # 初始化模块: 输入 1 通道, 输出 2 通道, 3x3 卷积核, Stride=1, Padding=1
    conv = Conv2DNumPy(in_channels=1, out_channels=2, kernel_size=3, stride=1, padding=1)
    relu = ReLUNumPy()
    
    # 前向传播
    conv_out = conv.forward(x_input)
    relu_out = relu.forward(conv_out)
    
    print("\n=== Conv2D 输出张量 Shape ===")
    print(conv_out.shape)  # 应输出 (1, 2, 5, 5)
    
    print("\n=== ReLU 输出数值结果 (前 3x3 区域) ===")
    print(relu_out[0, 0, :3, :3])

=== 输入张量 X Shape ===
(1, 1, 5, 5)

=== Conv2D 输出张量 Shape ===
(1, 2, 5, 5)

=== ReLU 输出数值结果 (前 3x3 区域) ===
[[0.         0.         0.23890546]
 [0.         0.         0.00792412]
 [0.23570777 0.         0.        ]]
